# The recipe as cooked: what do home cooks change, and where?

Review analysis for the 02805 project.

**Before running:** this notebook loads the cleaned data made by `project_Rakel.ipynb`. Run that notebook once first (it creates the folder `data/`), and again if the preprocessing there is changed.

In [ ]:
import numpy as np
import pandas as pd

recipes = pd.read_parquet("data/recipes_clean.parquet")
reviews = pd.read_parquet("data/reviews_clean.parquet")
communities = pd.read_parquet("data/communities.parquet")

# parquet gives lists back as numpy arrays, so turn them into normal Python lists again
for col in ["ingredients", "ingr_clean", "tags", "steps", "nutrition"]:
    recipes[col] = recipes[col].apply(list)

print(f"Recipes:     {len(recipes):,}")
print(f"Reviews:     {len(reviews):,}")
print(f"Communities: {communities['community'].nunique()} (broad: {communities['community_broad'].nunique()})")

In [ ]:
reviews.head()

## Test run: can the reviews be used?

Before building anything, we check four things:

1. What do the texts look like, and how long are they?
2. How many reviews mention a change to the recipe?
3. Can we pull out *"X instead of Y"* and match both X and Y to our clean ingredients?
4. A first look at the hypothesis: do people change baking recipes less?

### 1. The texts

The reviews contain leftovers from the website's HTML, e.g. `&#039;` for an apostrophe and `<br/>` for a line break. We turn these back into normal text and lowercase everything (week 5).

In [ ]:
import html
import re

print("With HTML codes (&...;):", reviews["review"].str.contains(r"&\w+;|&#\d+;").sum())
print("With <br/>:", reviews["review"].str.contains("<br").sum())

def clean_text(text):
    text = html.unescape(html.unescape(text))   # twice, because some are encoded twice (&amp;#039;)
    text = re.sub(r"<[^>]+>", " ", text)          # remove <br/> and other tags
    return " ".join(text.lower().split())         # lowercase, one space between words

reviews["text"] = reviews["review"].apply(clean_text)
reviews["n_words"] = reviews["text"].str.split().str.len()

print(reviews["n_words"].describe().round(0))

### 2. How many reviews mention a change?

A first, simple measurement: does the review contain words that people use when they change a recipe? This will miss some changes and wrongly catch others (e.g. *"I would not change a thing"*), so we check it by hand later.

In [ ]:
CHANGE_WORDS = {
    "instead of": r"\binstead of\b",
    "substitute": r"\bsubstitut",
    "in place of": r"\bin place of\b",
    "replace": r"\breplac",
    "added / I add": r"\badded\b|\bi add\b",
    "left out / omitted": r"\bleft out\b|\bomit|\bleave out\b",
    "more / less / extra": r"\b(?:more|less|extra)\b",
    "as written / no changes": r"as written|exactly as|no changes|wouldn't change|would not change",
}

for label, pattern in CHANGE_WORDS.items():
    print(f"{label:25s} {reviews['text'].str.contains(pattern).mean():6.1%}")

change_pattern = r"\binstead of\b|\bsubstitut|\bin place of\b|\breplac|\badded\b|\bi add\b|\bleft out\b|\bomit|\bleave out\b"
reviews["mentions_change"] = reviews["text"].str.contains(change_pattern)
print(f"\nReviews that mention a change (first 5 kinds): {reviews['mentions_change'].mean():.1%}")

### 3. Can we pull out *X instead of Y*?

We cut out the words just before and after *instead of*, and look for our clean ingredient names (from `recipes["ingr_clean"]`) in both pieces. If a piece contains several names, we take the longest (so *ground chicken* wins over *chicken*).

A strong automatic check: **Y (the ingredient that was replaced) should be in the recipe's own ingredient list.**

In [ ]:
vocabulary = sorted(set(x for xs in recipes["ingr_clean"] for x in xs), key=len, reverse=True)
vocab_pattern = re.compile(r"\b(" + "|".join(re.escape(v) for v in vocabulary) + r")s?\b")

def find_ingredient(piece):
    found = vocab_pattern.findall(piece)
    return max(found, key=len) if found else None

rows = []
for r in reviews[reviews["text"].str.contains("instead of")].itertuples():
    for m in re.finditer(r"([\w' -]{1,40}?) instead of (?:the |using |adding )?([\w' -]{1,40})", r.text):
        rows.append({"recipe_id": r.recipe_id, "x_text": m.group(1), "y_text": m.group(2),
                     "x": find_ingredient(m.group(1)), "y": find_ingredient(m.group(2))})

subs = pd.DataFrame(rows)
ingr_of = dict(zip(recipes["id"], recipes["ingr_clean"]))
subs["y_in_recipe"] = [y in ingr_of[r] for y, r in zip(subs["y"], subs["recipe_id"])]

both = subs.dropna(subset=["x", "y"])
both = both[both["x"] != both["y"]]
print(f"'instead of' found:              {len(subs):,}")
print(f"Both X and Y matched an ingredient: {len(both):,} ({len(both) / len(subs):.0%})")
print(f"... and Y is in the recipe:          {both['y_in_recipe'].sum():,} ({both['y_in_recipe'].mean():.0%} of those)")
print(f"Different substitution pairs:        {both[['x', 'y']].drop_duplicates().shape[0]:,}")

Examples, and the most common substitutions:

In [ ]:
for r in both.sample(12, random_state=1).itertuples():
    print(f"{r.x:20s} instead of {r.y:20s} | in recipe: {r.y_in_recipe!s:5s} | \"{r.x_text} instead of {r.y_text}\"")

In [ ]:
# most common substitutions: "x instead of y" (only where y is in the recipe)
both[both["y_in_recipe"]].groupby(["x", "y"]).size().sort_values(ascending=False).head(25)

### 4. A first look: do people change baking recipes less?

Share of reviews that mention a change, in the 3 broad communities. **Not tested yet**: this is only to see whether the idea is worth pursuing.

In [ ]:
reviews = reviews.merge(communities, left_on="recipe_id", right_on="id", how="left")

broad_names = {}
for c in sorted(communities["community_broad"].unique()):
    ids = communities.loc[communities["community_broad"] == c, "id"]
    top = pd.Series([x for xs in recipes.loc[recipes["id"].isin(ids), "ingr_clean"] for x in xs]).value_counts().index[:4]
    broad_names[c] = ", ".join(top)

summary = reviews.groupby("community_broad")["mentions_change"].agg(["mean", "size"])
summary.index = [broad_names[c] for c in summary.index]
summary.columns = ["share mentioning a change", "reviews"]
summary.round(3)

### What the test run shows

- **The texts are usable.** Median 43 words per review. About 19,000 contained HTML codes, now cleaned.
- **Changes are common.** 26.5 % of the reviews use one of the change words (*instead of, substitute, replace, added, left out*). 7.7 % contain *instead of*.
- **Substitutions can be extracted.** 25,038 *instead of*; in 11,802 both sides match one of our ingredients, and in 7,904 of those the replaced ingredient is really in the recipe. That gives about 5,000 different substitution pairs, enough for a substitution network.
- **The most common substitutions make sense:** butter instead of margarine, olive oil instead of butter, turkey instead of beef, chicken broth instead of water, honey instead of sugar, yogurt instead of sour cream.
- **Errors we can see:** *coconut water* becomes *coconut* (the name is not in our vocabulary), and *butterscotch pudding instead of vanilla [pudding]* loses the word *pudding* on the right side. The check *Y is in the recipe* removes many of these. We need a hand-labelled sample to measure the precision.
- **First look at the hypothesis (not tested):** savoury 28.6 %, Asian-style 25.8 %, baking 23.5 % of reviews mention a change. The direction fits *baking is chemistry*, but we still need a null model, and to check that it is not just a few recipes or users driving it.

## Hand-labelled samples (validation)

Our two measurements (*mentions a change* and *X instead of Y*) must be checked against what a person reads in the review (week 7–8). This cell makes two sample files in `labels/`, which we fill in by hand:

1. **`labels/change_sample.csv`**: 100 random reviews. In the column `label_change`, write **1** if the review describes a change the cook made to the recipe (added, left out, replaced, more/less of something), and **0** if not. The automatic answer is *not* in the file, so it cannot influence us.
2. **`labels/substitution_sample.csv`**: 50 random extracted substitutions. In `x_ok` and `y_ok`, write **1** if X (the new ingredient) or Y (the replaced ingredient) is correct, and **0** if not. If it is wrong, write the right ingredient in `x_should_be` / `y_should_be`.

The files are only created if they do not exist yet, so running the notebook again never overwrites our labels. `labels/` is on GitHub, so we can share the work.

In [ ]:
import os

os.makedirs("labels", exist_ok=True)

if not os.path.exists("labels/change_sample.csv"):
    sample = reviews.sample(100, random_state=1)[["user_id", "recipe_id", "text"]]
    sample["label_change"] = ""
    sample.to_csv("labels/change_sample.csv", index=False, encoding="utf-8-sig")
    print("Created labels/change_sample.csv")

if not os.path.exists("labels/substitution_sample.csv"):
    sample = both.sample(50, random_state=1)[["recipe_id", "x_text", "y_text", "x", "y"]].copy()
    sample["x_ok"], sample["y_ok"], sample["x_should_be"], sample["y_should_be"] = "", "", "", ""
    sample.to_csv("labels/substitution_sample.csv", index=False, encoding="utf-8-sig")
    print("Created labels/substitution_sample.csv")

When the files are filled in, this cell compares our labels with the automatic measurements.

In [ ]:
change = pd.read_csv("labels/change_sample.csv")
if change["label_change"].notna().all():
    change = change.merge(reviews[["user_id", "recipe_id", "mentions_change"]], on=["user_id", "recipe_id"])
    tp = (change["mentions_change"] & (change["label_change"] == 1)).sum()
    print(f"Change words, precision: {tp / change['mentions_change'].sum():.0%} "
          f"(of the reviews we flag, how many really describe a change)")
    print(f"Change words, recall:    {tp / (change['label_change'] == 1).sum():.0%} "
          f"(of the reviews that describe a change, how many we flag)")
else:
    print("labels/change_sample.csv is not filled in yet")

subs_labels = pd.read_csv("labels/substitution_sample.csv")
if subs_labels[["x_ok", "y_ok"]].notna().all().all():
    print(f"Substitutions: X correct {subs_labels['x_ok'].mean():.0%}, Y correct {subs_labels['y_ok'].mean():.0%}, "
          f"both correct {((subs_labels['x_ok'] == 1) & (subs_labels['y_ok'] == 1)).mean():.0%}")
else:
    print("labels/substitution_sample.csv is not filled in yet")